# Rainfall Forecasting: Changi, Singapore (1982 to 2020)

**Project:** PRAICP-1004 | **Data:** NEA Changi Climate Station via data.gov.sg | **Frequency:** monthly

## Goal
1. Forecast **monthly total rainfall (mm)** for the next 12 months (Jul 2020 to Jun 2021).
2. Forecast the **highest daily rainfall in each month (mm)** for the same 12 months.

## Data (3 files in `../data/`)
| File | Column | Meaning |
|---|---|---|
| `rainfall-monthly-total.csv` | `total_rainfall` | Total rainfall in the month (mm) |
| `rainfall-monthly-highest-daily-total.csv` | `maximum_rainfall_in_a_day` | Highest single-day rainfall in the month (mm) |
| `rainfall-monthly-number-of-rain-days.csv` | `no_of_rainy_days` | Days with at least 0.2 mm of rain |

## Notebook map
| Step | What happens |
|---|---|
| 0 | Setup: imports and folder paths |
| 1, 1b | Load the data and run sanity checks |
| 2 | Plot the three series |
| 3 | Seasonality, trend and correlation (EDA) |
| 4 | Time-ordered train/test split |
| 5 | Baseline models |
| 6, 7, 8 | SARIMA and Prophet on total rainfall, then comparison |
| 9 | SARIMA and Prophet on highest daily rainfall |
| 10 | Rolling validation (5 test years) |
| 11 | Final forecast with 95% intervals |
| 12 | Results table and project summary |

## How to run
Start Jupyter from the repo root or from `notebooks/`. The paths in Step 0 handle both. In Colab, uncomment the `!pip install` line in Step 0.

In [ ]:
# Step 0: setup
# In Colab, uncomment the next line:
# !pip install prophet statsmodels -q

import itertools, logging, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet

warnings.filterwarnings("ignore")                      # hides SARIMAX convergence warnings during the grid search
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

# Project paths: work whether Jupyter starts in notebooks/ or in the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA      = ROOT / "data"
RESULTS   = ROOT / "outputs" / "results"
FORECASTS = ROOT / "outputs" / "forecasts"
IMAGES    = ROOT / "images"
for p in (RESULTS, FORECASTS, IMAGES):
    p.mkdir(parents=True, exist_ok=True)

TOTAL, MAXD, DAYS = "total_rainfall", "maximum_rainfall_in_a_day", "no_of_rainy_days"
print("Project root:", ROOT)

### Step 0 explained: setup
**What we did**
- Imported the libraries: `pandas`/`numpy` for data, `matplotlib` for plots, `scikit-learn` for error metrics, `statsmodels` for SARIMA and decomposition, `prophet` for the second forecasting model.
- Defined folder paths once (`DATA`, `RESULTS`, `FORECASTS`, `IMAGES`) so every file the notebook saves lands in the right folder of the repo.
- Silenced warnings. SARIMA's grid search fits 36 candidate models and some print *ConvergenceWarning* messages. These are common with seasonal models and do not stop the best model from being usable.

**Why:** a single setup cell keeps the rest of the notebook clean and makes the project reproducible on any machine.

In [ ]:
# Step 1: load the three files into one table
def load(path):
    return pd.read_csv(path, parse_dates=["month"], index_col="month")

df = (load(DATA / "rainfall-monthly-total.csv")
      .join(load(DATA / "rainfall-monthly-highest-daily-total.csv"))
      .join(load(DATA / "rainfall-monthly-number-of-rain-days.csv")))
df = df.asfreq("MS")       # explicit month-start frequency, required by statsmodels

print(df.shape)            # expect (462, 3)
display(df.head()); display(df.tail())
print(df.isna().sum())     # expect zeros
display(df.describe())

### Step 1 explained: load and inspect
**What we did**
- Read the three CSVs. Each has a `month` column (`YYYY-MM`) that we parse as a date and use as the index.
- Joined them on that index into one table with three columns, then set the frequency to month-start (`"MS"`).

**What we found**
- **462 rows x 3 columns**, from **Jan 1982 to Jun 2020**, with **no missing values**. So no cleaning or interpolation was needed.
- Mean monthly rainfall is about **176 mm** (median 158 mm, max 766 mm). The mean is above the median, so the data is **right-skewed** (a few very wet months).
- The standard deviation (113 mm) is about 64% of the mean, so rainfall is **very variable** month to month.
- The highest daily rainfall reaches **216 mm**, far above its 75th percentile of 63 mm, so extreme days exist.
- Rain days per month range from 1 to 27.

In [ ]:
# Step 1b: sanity checks
# 1. The highest single day can never exceed the monthly total
print("Rows where max-daily > total:", (df[MAXD] > df[TOTAL]).sum())   # expect 0

# 2. Extreme months
print("Wettest month:", df[TOTAL].idxmax().strftime("%Y-%m"), df[TOTAL].max(), "mm")
print("Driest month: ", df[TOTAL].idxmin().strftime("%Y-%m"), df[TOTAL].min(), "mm")
print("Highest daily record:", df[MAXD].idxmax().strftime("%Y-%m"), df[MAXD].max(), "mm")
print("Months with < 10 mm total:", (df[TOTAL] < 10).sum())
print("Skewness of total rainfall:", round(df[TOTAL].skew(), 2))

# 3. Outliers
df[TOTAL].plot(kind="box", vert=False, title="Total rainfall: outliers", figsize=(8, 2.5))
plt.show()

### Step 1b explained: sanity checks
**What we did:** checked that the data makes physical sense before modelling.

**What we found**
- **0 rows** where the highest daily rainfall exceeds the monthly total, so the files are consistent with each other.
- **Wettest month: Dec 2006 (765.9 mm).** **Driest month: Feb 2014 (0.2 mm).** **Highest daily record: Jan 2011 (216.2 mm).**
- Only **5 months** have less than 10 mm, and the **skewness is 1.24** (right-skewed).
- The box plot shows many high outliers. We keep them, because they are real weather events, not errors.

**Why it matters:** the skew and outliers warn us that squared-error metrics such as RMSE will be dominated by a few wet months. This is why we report both **MAE** and **RMSE** later.

In [ ]:
# Step 2: plot the three series
axes = df.plot(subplots=True, figsize=(12, 8), title="Changi monthly rainfall")
plt.savefig(IMAGES / "eda_series.png", dpi=150, bbox_inches="tight")
plt.show()

### Step 2 explained: look at the series
**What we did:** plotted total rainfall, highest daily rainfall and rain days over time.

**What we saw**
- **Total rainfall:** no obvious long-term rise or fall, just large month-to-month swings. The biggest spike is the Dec 2006 event (766 mm).
- **Highest daily rainfall:** mostly in the 25 to 75 mm range, with occasional large spikes. These outliers make it harder to forecast.
- **Rain days:** looks slightly higher in later years, but this is not clear by eye. Step 3 checks it numerically.

Saved as `images/eda_series.png`.

In [ ]:
# Step 3: seasonality, trend and correlation
# (a) Decomposition: trend + seasonal + residual, with a 12-month cycle
decomp = seasonal_decompose(df[TOTAL], model="additive", period=12)
fig = decomp.plot(); fig.set_size_inches(12, 8)
fig.savefig(IMAGES / "eda_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()

# (b) Average rainfall for each calendar month
monthly_avg = df.groupby(df.index.month)[TOTAL].mean()
ax = monthly_avg.plot(kind="bar", figsize=(10, 4), title="Average total rainfall by calendar month (1982 to 2020)")
ax.set_xlabel("Month"); ax.set_ylabel("mm")
plt.savefig(IMAGES / "eda_monthly_avg.png", dpi=150, bbox_inches="tight")
plt.show()
print(monthly_avg.round(1).to_dict())

# (c) Yearly totals (2020 excluded because it only has 6 months)
full = df[df.index.year < 2020]
yearly = full.groupby(full.index.year)[TOTAL].sum()
yearly.plot(title="Total rainfall per year", figsize=(10, 3.5)); plt.ylabel("mm"); plt.show()
print("Average annual rainfall: %.0f mm | trend: %.1f mm per year" % (yearly.mean(), np.polyfit(yearly.index, yearly.values, 1)[0]))

days = full.groupby(full.index.year)[DAYS].sum()
print("Rain days per year: 1982-91 average %.0f vs 2010-19 average %.0f" % (days.loc[1982:1991].mean(), days.loc[2010:2019].mean()))

# (d) Correlation between the three series
display(df.corr().round(2))

### Step 3 explained: seasonality, trend, correlation
**What we did**
- **Decomposition** splits total rainfall into *trend*, *seasonal* and *residual* parts, assuming a 12-month cycle.
- Computed the **average rainfall per calendar month**, **yearly totals**, and the **correlation** between the three series.

**What we found**
- **Strong seasonality:** the wettest months are **December (about 318 mm)**, **November (about 256 mm)** and **January (about 226 mm)**, which matches Singapore's northeast monsoon. **February is the driest (about 105 mm)**, and March to October sit between roughly 134 and 166 mm.
- **Weak trend:** annual rainfall averages about **2,118 mm** (range 1,119 to 2,886 mm). The fitted line slopes slightly downward (about -4 mm per year), but yearly totals vary by a few hundred mm, so this is small and we did not test it for significance.
- **Rain days:** the yearly count averages about **161 days in 1982 to 1991** and **173 days in 2010 to 2019**, a modest rise.
- **Correlations:** total rainfall vs highest daily = **0.80**, total vs rain days = **0.70**, highest daily vs rain days = **0.41**. Total rainfall depends on both how often it rains and how hard it rains at peak.

**Modelling decision:** a clear yearly cycle plus no strong trend is a good fit for **seasonal** models. We use SARIMA with seasonal differencing (period 12) and Prophet with yearly seasonality.

In [ ]:
# Step 4: time-ordered train/test split (never shuffle time series)
train = df.iloc[:-12]     # Jan 1982 to Jun 2019
test  = df.iloc[-12:]     # Jul 2019 to Jun 2020

print("Train:", train.shape, train.index.min().date(), "to", train.index.max().date())
print("Test: ", test.shape,  test.index.min().date(),  "to", test.index.max().date())

### Step 4 explained: train/test split
**What we did:** held out the **last 12 months** as the test set and kept everything before it for training.

**Result:** train = **450 months** (Jan 1982 to Jun 2019), test = **12 months** (Jul 2019 to Jun 2020).

**Why no shuffling:** in a time series each month depends on the months before it. Shuffling would let the model "see the future" while training, giving fake, over-optimistic scores.

**Caveat:** one 12-month test window is small and noisy. Step 10 fixes this with rolling validation across 5 windows.

In [ ]:
# Step 5: baselines (the bar every real model must beat)
def score(actual, pred):
    mae  = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    return mae, rmse

def baselines(col):
    r = {}
    # Seasonal naive: repeat the last 12 months of the training data
    r["Seasonal naive (last year)"] = score(test[col], train[col].iloc[-12:].values)
    # Monthly average: for each calendar month, use its historical training average
    month_avg = train[col].groupby(train.index.month).mean()
    r["Monthly average"] = score(test[col], month_avg.loc[test.index.month].values)
    return r

res_total = baselines(TOTAL)
res_max   = baselines(MAXD)

print("TOTAL RAINFALL");           display(pd.DataFrame(res_total, index=["MAE", "RMSE"]).T)
print("HIGHEST DAILY RAINFALL");   display(pd.DataFrame(res_max,   index=["MAE", "RMSE"]).T)

### Step 5 explained: baselines
**What we did:** built two simple forecasts and scored them on the test year with **MAE** (average error in mm) and **RMSE** (like MAE but punishes big misses harder). Lower is better for both.
- **Seasonal naive:** "next year looks like last year."
- **Monthly average:** "each month looks like its historical average."

**Results (test year Jul 2019 to Jun 2020)**

| Total rainfall | MAE | RMSE |
|---|---|---|
| Seasonal naive | 87.98 | 112.54 |
| Monthly average | 92.66 | 102.14 |

| Highest daily rainfall | MAE | RMSE |
|---|---|---|
| Seasonal naive | 22.47 | 27.30 |
| Monthly average | 19.73 | 26.55 |

**Takeaway:** the two baselines disagree on total rainfall (seasonal naive has the better MAE, the monthly average has the better RMSE). The monthly average is the stronger overall baseline, because copying last year's noise produces a few large misses. Any model we build must beat these numbers to justify its complexity.

In [ ]:
# Step 6: SARIMA on total rainfall
# SARIMA(p,0,q)(P,1,Q,12): seasonal differencing (D=1) removes the yearly cycle.
# We search small orders and keep the model with the lowest AIC (a fit-vs-complexity score).
def fit_sarima(series):
    best, best_aic = (1, 1, 1, 1), np.inf
    for p, q, P, Q in itertools.product([0, 1, 2], [0, 1, 2], [0, 1], [0, 1]):
        try:
            m = SARIMAX(series, order=(p, 0, q), seasonal_order=(P, 1, Q, 12)).fit(disp=False)
            if m.aic < best_aic:
                best, best_aic = (p, q, P, Q), m.aic
        except Exception:
            pass
    p, q, P, Q = best
    model = SARIMAX(series, order=(p, 0, q), seasonal_order=(P, 1, Q, 12)).fit(disp=False)
    return model, best

def to_orders(o):                       # (p,q,P,Q) -> (order, seasonal_order)
    return (o[0], 0, o[1]), (o[2], 1, o[3], 12)

sar, orders = fit_sarima(train[TOTAL])
p_sar = sar.forecast(12).clip(lower=0).values      # rainfall cannot be negative

res_total["SARIMA"] = score(test[TOTAL], p_sar)
print("Best (p,q,P,Q):", orders)
display(pd.DataFrame(res_total, index=["MAE", "RMSE"]).T.sort_values("RMSE"))

### Step 6 explained: SARIMA on total rainfall
**What we did**
- Tried 36 SARIMA configurations (`p, q` in 0 to 2; `P, Q` in 0 to 1) with seasonal differencing at period 12, and kept the one with the **lowest AIC**.
- Forecast the 12 test months and clipped negatives to 0 (rainfall can't be negative).

**Result:** best orders **(p,q,P,Q) = (2,1,0,1)**, i.e. SARIMA(2,0,1)(0,1,1,12).

| Model | MAE | RMSE |
|---|---|---|
| **SARIMA** | **86.80** | **97.46** |
| Monthly average | 92.66 | 102.14 |
| Seasonal naive | 87.98 | 112.54 |

**Takeaway:** SARIMA has the lowest RMSE and MAE, but its MAE edge over seasonal naive is only about 1 mm, which is noise on a 12-month test. The convergence warnings come from some of the 36 candidates in the search and are not a problem for the selected model.

In [ ]:
# Step 7: Prophet on total rainfall
def fit_prophet_forecast(series, steps=12):
    d = series.reset_index(); d.columns = ["ds", "y"]
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False).fit(d)
    fut = m.make_future_dataframe(periods=steps, freq="MS")
    return m.predict(fut).tail(steps)["yhat"].values

p_pro = np.clip(fit_prophet_forecast(train[TOTAL]), 0, None)

res_total["Prophet"] = score(test[TOTAL], p_pro)
display(pd.DataFrame(res_total, index=["MAE", "RMSE"]).T.sort_values("RMSE"))

### Step 7 explained: Prophet on total rainfall
**What we did:** fitted Facebook's **Prophet** (trend plus yearly seasonality) on the training data and forecast the same 12 months.

**Result**

| Model | MAE | RMSE |
|---|---|---|
| SARIMA | 86.80 | 97.46 |
| Monthly average | 92.66 | 102.14 |
| Prophet | 96.74 | 103.63 |
| Seasonal naive | 87.98 | 112.54 |

**Takeaway:** Prophet did **not** beat the monthly average (RMSE 103.6 vs 102.1, MAE 96.7 vs 92.7). It found no structure beyond a repeating yearly pattern. This is plausible: Singapore's monthly rainfall is noisy with weak seasonality and no trend, so a more flexible model has little extra to learn. A fancier model is not automatically a better one.

In [ ]:
# Step 8: comparison table and plot for total rainfall
table_total = pd.DataFrame(res_total, index=["MAE", "RMSE"]).T.sort_values("RMSE")
table_total.to_csv(RESULTS / "results_total.csv")
display(table_total)

plt.figure(figsize=(12, 4))
plt.plot(train[TOTAL][-36:], label="train (last 3 years)")
plt.plot(test[TOTAL], label="actual", linewidth=2, color="black")
plt.plot(test.index, p_sar, label="SARIMA")
plt.plot(test.index, p_pro, label="Prophet")
plt.title("Total rainfall: test year (Jul 2019 to Jun 2020)"); plt.ylabel("mm"); plt.legend()
plt.savefig(IMAGES / "total_test.png", dpi=150, bbox_inches="tight")
plt.show()

### Step 8 explained: compare and save
**What we did:** put all four models into one table sorted by RMSE, saved it to `outputs/results/results_total.csv`, and plotted the forecasts against the actual test-year values (`images/total_test.png`).

**How to read the plot:** the actual line swings up and down by month, while both model forecasts are much smoother. They capture the seasonal level (wetter around Nov to Jan) but cannot predict individual wet or dry months. That is a real limit of this data, not a bug.

**Ranking on total rainfall (test year):** SARIMA, then monthly average, then Prophet, then seasonal naive (by RMSE).

In [ ]:
# Step 9: same models on the highest daily rainfall (second project goal)
sar_m, orders_m = fit_sarima(train[MAXD])
pm_sar = sar_m.forecast(12).clip(lower=0).values
pm_pro = np.clip(fit_prophet_forecast(train[MAXD]), 0, None)

res_max["SARIMA"]  = score(test[MAXD], pm_sar)
res_max["Prophet"] = score(test[MAXD], pm_pro)

table_max = pd.DataFrame(res_max, index=["MAE", "RMSE"]).T.sort_values("RMSE")
table_max.to_csv(RESULTS / "results_max.csv")
print("Best (p,q,P,Q):", orders_m)
display(table_max)

plt.figure(figsize=(12, 4))
plt.plot(train[MAXD][-36:], label="train (last 3 years)")
plt.plot(test[MAXD], label="actual", linewidth=2, color="black")
plt.plot(test.index, pm_sar, label="SARIMA")
plt.plot(test.index, pm_pro, label="Prophet")
plt.title("Highest daily rainfall: test year (Jul 2019 to Jun 2020)"); plt.ylabel("mm"); plt.legend()
plt.savefig(IMAGES / "max_test.png", dpi=150, bbox_inches="tight")
plt.show()

### Step 9 explained: highest daily rainfall
**What we did:** repeated the SARIMA grid search and the Prophet fit on the **highest daily rainfall** series, saved the table to `outputs/results/results_max.csv` and the plot to `images/max_test.png`.

**Result:** best orders **(p,q,P,Q) = (1,1,0,1)**, i.e. SARIMA(1,0,1)(0,1,1,12).

| Model | MAE | RMSE |
|---|---|---|
| **SARIMA** | **18.69** | **24.34** |
| Monthly average | 19.73 | 26.55 |
| Seasonal naive | 22.47 | 27.30 |
| Prophet | 21.71 | 28.46 |

**Takeaway:** SARIMA is best, about 8% better on RMSE and 5% on MAE than the monthly average. Prophet is the worst on RMSE, so we drop it for the rest of the project. A 5 to 8% gap on one 12-month window could still be luck, which is why Step 10 exists.

In [ ]:
# Step 10: rolling validation, 5 test years (Jul 2015 to Jun 2020), model refit on each fold
def rolling_eval(col, order, seasonal_order, folds=5):
    rows = []
    for k in range(folds, 0, -1):
        end = len(df) - 12 * (k - 1)
        tr, te = df[col].iloc[:end - 12], df[col].iloc[end - 12:end]

        p_s = SARIMAX(tr, order=order, seasonal_order=seasonal_order).fit(disp=False).forecast(12).clip(lower=0).values
        p_a = tr.groupby(tr.index.month).mean().loc[te.index.month].values
        p_n = tr.iloc[-12:].values

        for name, p in [("SARIMA", p_s), ("Monthly average", p_a), ("Seasonal naive", p_n)]:
            mae, rmse = score(te, p)
            rows.append((name, k, mae, rmse))
    out = pd.DataFrame(rows, columns=["model", "fold", "MAE", "RMSE"])
    return out.groupby("model")[["MAE", "RMSE"]].mean().sort_values("RMSE")

cv_total = rolling_eval(TOTAL, *to_orders(orders))
cv_max   = rolling_eval(MAXD,  *to_orders(orders_m))

print("TOTAL RAINFALL (5-fold rolling average)");         display(cv_total)
print("HIGHEST DAILY RAINFALL (5-fold rolling average)"); display(cv_max)
cv_total.to_csv(RESULTS / "cv_total.csv"); cv_max.to_csv(RESULTS / "cv_max.csv")

### Step 10 explained: rolling validation
**What we did:** instead of trusting one test year, we tested on **5 consecutive 12-month windows** (Jul 2015 to Jun 2020). For each window the models only saw data **before** it, and the SARIMA model was refit. We then averaged the errors.

**Results (average over 5 folds)**

| Total rainfall | MAE | RMSE |
|---|---|---|
| **SARIMA** | **64.73** | **75.54** |
| Monthly average | 66.33 | 76.75 |
| Seasonal naive | 78.33 | 97.26 |

| Highest daily rainfall | MAE | RMSE |
|---|---|---|
| **SARIMA** | **20.21** | **23.89** |
| Monthly average | 20.72 | 24.77 |
| Seasonal naive | 21.98 | 26.10 |

**Takeaways**
- SARIMA wins on both targets, so its single-year lead was not luck.
- The margin over the monthly average is **small** (about 1.6% on RMSE for total rainfall, about 3.5% for highest daily). Most variation is unpredictable noise, and mainly the seasonal average is reliably forecastable.
- Seasonal naive is clearly the weakest.

**Limitations to state honestly:** Prophet was not included in the rolling validation (it was already worse on the single test year). The SARIMA orders were chosen on data up to Jun 2019, which overlaps some validation windows, so these scores may be slightly optimistic.

In [ ]:
# Step 11: final forecast, Jul 2020 to Jun 2021 (refit SARIMA on ALL 462 months)
def final_forecast(col, o):
    order, seasonal_order = to_orders(o)
    m = SARIMAX(df[col], order=order, seasonal_order=seasonal_order).fit(disp=False)
    fc = m.get_forecast(12)
    out = fc.predicted_mean.clip(lower=0).to_frame("forecast")
    out[["lower", "upper"]] = fc.conf_int().clip(lower=0).values      # 95% interval
    return out

f_total = final_forecast(TOTAL, orders)
f_max   = final_forecast(MAXD,  orders_m)
f_total.to_csv(FORECASTS / "forecast_total_next_12_months.csv")
f_max.to_csv(FORECASTS / "forecast_max_daily_next_12_months.csv")

print("TOTAL RAINFALL FORECAST (mm)");           display(f_total.round(1))
print("HIGHEST DAILY RAINFALL FORECAST (mm)");   display(f_max.round(1))

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
for ax, (col, f, title) in zip(axes, [
        (TOTAL, f_total, "Total rainfall: forecast Jul 2020 to Jun 2021"),
        (MAXD,  f_max,   "Highest daily rainfall: forecast Jul 2020 to Jun 2021")]):
    ax.plot(df[col][-60:], label="history (last 5 years)")
    ax.plot(f["forecast"], label="forecast", color="red")
    ax.fill_between(f.index, f["lower"], f["upper"], color="red", alpha=0.15, label="95% interval")
    ax.set_title(title); ax.set_ylabel("mm"); ax.legend()
plt.tight_layout()
plt.savefig(IMAGES / "final_forecast.png", dpi=150, bbox_inches="tight")
plt.show()

### Step 11 explained: final forecast
**What we did:** refit the winning model (SARIMA) on **all** available data (Jan 1982 to Jun 2020) and forecast the next 12 months for both targets, with a **95% prediction interval**. Negative values are clipped to 0. Results are saved in `outputs/forecasts/` and the plot in `images/final_forecast.png`. The Streamlit app reads these two CSVs.

**What the forecast shows**
- **Total rainfall** is expected to peak in **Dec 2020 at about 337 mm** (interval about 142 to 532 mm), after a rise from Oct. The lowest forecast months are Sep 2020 (about 125 mm) and Feb 2021 (about 123 mm).
- **Highest daily rainfall** also peaks in **Dec 2020 at about 77 mm**, with the lowest forecast in Feb 2021 (about 35 mm).
- The shape matches the northeast-monsoon pattern found in Step 3.

**Important:** the intervals are **very wide** (for example 0 to 330 mm for July 2020). That is the honest message: the model gets the seasonal level right, but single months stay highly uncertain.

In [ ]:
# Step 12: consolidated results, read back from the saved files
rt = pd.read_csv(RESULTS / "results_total.csv", index_col=0).round(2)
rm = pd.read_csv(RESULTS / "results_max.csv",   index_col=0).round(2)
ct = pd.read_csv(RESULTS / "cv_total.csv",      index_col=0).round(2)
cm = pd.read_csv(RESULTS / "cv_max.csv",        index_col=0).round(2)

print("Single test year, total rainfall");        display(rt)
print("Single test year, highest daily rainfall"); display(rm)
print("5-fold rolling validation, total rainfall");        display(ct)
print("5-fold rolling validation, highest daily rainfall"); display(cm)

### Step 12 explained: consolidated results
Reads the four saved result files back so the whole project's numbers sit in one place and match what the README and the app use.

# Project Summary

**Problem.** Forecast Changi (Singapore) monthly total rainfall and monthly highest daily rainfall for Jul 2020 to Jun 2021, using 462 months of NEA data (Jan 1982 to Jun 2020).

**Data and checks.** Three clean monthly series with no missing values. Highest daily rainfall never exceeds the monthly total. Rainfall is right-skewed (skewness 1.24) and highly variable (standard deviation about 64% of the mean), with extreme events such as Dec 2006 (766 mm in a month) and Jan 2011 (216 mm in a day).

**Key EDA findings.**
1. Strong seasonality: wettest in **Dec, Nov, Jan** (about 318, 256, 226 mm on average), driest in **Feb** (about 105 mm), consistent with the northeast monsoon.
2. No strong long-term trend in annual rainfall (average about 2,118 mm per year). Rain days per year rose modestly, from about 161 (1982 to 1991) to about 173 (2010 to 2019).
3. Total rainfall correlates strongly with both highest daily rainfall (0.80) and rain days (0.70).

**Method.** Time-ordered split (last 12 months as test), two baselines (seasonal naive, monthly average), SARIMA with an AIC grid search, Prophet, and 5-fold rolling validation. Forecasts were clipped at 0 and given 95% intervals.

**Results.**

| Rolling validation (5 years), RMSE | SARIMA | Monthly average | Seasonal naive |
|---|---|---|---|
| Total rainfall (mm) | **75.5** | 76.7 | 97.3 |
| Highest daily rainfall (mm) | **23.9** | 24.8 | 26.1 |

- **SARIMA is the best model** for both targets, consistently across 5 test years. Prophet did not beat the simple monthly-average baseline.
- The edge over the monthly average is small (about 2 to 4%). Most month-to-month variation is unpredictable; the reliable signal is the seasonal pattern.

**Forecast (Jul 2020 to Jun 2021).** Rainfall rises from Oct and peaks in **Dec 2020** (about 337 mm total, about 77 mm highest daily), with the lowest months around Sep 2020 and Feb 2021. 95% intervals are wide.

**Limitations.** One station only; 38 years of data; no external predictors; the data ends in Jun 2020; single-month extremes cannot be predicted; the SARIMA orders were chosen on data that overlaps some validation windows; Prophet was not part of rolling validation.

**Next steps.** Add rain days (or climate indices such as ENSO and the Indian Ocean Dipole) as exogenous variables in SARIMAX; try an LSTM or gradient-boosting comparison; forecast the number of rain days; model extremes separately (for example with extreme value theory); compare several stations.

**Deliverables.** This notebook, a Streamlit forecast app (`app.py`), forecast CSVs, result tables, and plots.